# 06. Улучшение модели Semantic News Novelty (v3)

Цель ноутбука — cоставить логичную цепочку экспериментов без дублирования большого служебного кода.

Порядок работы:

1. Воспроизводим baseline `exp_00b`.
2. Отдельно выбираем кластеризацию, не обучая final-step модель на каждом шаге.
3. Фиксируем кластеризацию `exp_10`, выбранную по silver-positive сигналу без использования golden.
4. На выбранной кластеризации сравниваем novelty classifiers.
5. Сохраняем выбранную финальную модель для `final_pipeline`.
6. Опционально проверяем дообученный BGE-M3 как ablation.

Основное методологическое решение: silver используется для кластеризации только как weak-positive источник. Пары из разных silver-кластеров не считаются надёжными negative, потому что silver оказался пере-фрагментированным.

## 1. Импорты и настройка путей

В ноутбуке остаётся только orchestration-код. Основная логика вынесена в пакет `model/`.

In [46]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "model").exists():
            return candidate
    raise RuntimeError(f"Project root with src/model was not found from {start}")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"

for path in [PROJECT_ROOT, SRC_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

%load_ext autoreload
%autoreload 2

from model.config import ExperimentPaths, SemanticNewsBaselineConfig, SignificanceModelConfig
from model.data import (
    load_clean_news,
    load_annotation,
    remove_train_eval_leakage,
    normalize_news_id,
)
from model.embeddings import SentenceTransformerEncoder
from model.features import LEGACY_SIGNIFICANCE_FEATURE_COLUMNS, build_legacy_significance_features
from model.significance_model import CatBoostSignificanceModel
from model.legacy_pipeline import LegacyNewsNoveltyPipeline, LegacyNewsPipelineConfig
from model.evaluation import evaluate_predictions, compact_metrics_table
from model.experiment_tracking import ExperimentTracker
from model.attach_clustering import (
    AttachClusteringConfig,
    BaselineClusteringConfig,
    SilverPositiveSelectionConfig,
    build_candidate_pairs,
    build_best_candidate_attach_clusters,
    evaluate_cluster_ids_on_reference,
    make_clustered_news,
    prepare_silver_positive_reference,
    run_silver_positive_attach_sweep,
    select_silver_positive_variant,
    get_attach_config_from_sweep_row,
)
from model.classifier_training import (
    make_significance_training_frame,
    train_validation_split,
    fit_catboost_binary,
    fit_mlp_binary,
    fit_logreg_binary,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [47]:
paths = ExperimentPaths(project_root=PROJECT_ROOT)
paths.predictions_dir.mkdir(parents=True, exist_ok=True)
paths.artifacts_dir.mkdir(parents=True, exist_ok=True)
paths.models_dir.mkdir(parents=True, exist_ok=True)

paths.clean_news_path, paths.golden_path, paths.silver_path

(WindowsPath('E:/ML/Projects/Git/news-flow-analysis/data/prepared/lenta_clean_news.csv'),
 WindowsPath('E:/ML/Projects/Git/news-flow-analysis/data/prepared/lenta_golden_set_human.csv'),
 WindowsPath('E:/ML/Projects/Git/news-flow-analysis/data/prepared/lenta_silver_set_llm.csv'))

## 2. Загрузка данных

`candidate_news` — полный candidate pool, на котором строится кластеризация. Golden используется для оценки, но не для выбора `exp_10`.

In [48]:
clean_news = load_clean_news(paths.clean_news_path)
golden = load_annotation(paths.golden_path)
silver_raw = load_annotation(paths.silver_path)
silver = remove_train_eval_leakage(silver_raw, golden, id_column="news_id")

candidate_pool_path = paths.prepared_dir / "lenta_golden_candidate_pool.csv"
if not candidate_pool_path.exists():
    raise FileNotFoundError(f"Не найден candidate pool: {candidate_pool_path}")

raw_candidate_pool = pd.read_csv(candidate_pool_path)

print("clean_news:", clean_news.shape)
print("candidate_pool:", raw_candidate_pool.shape)
print("golden:", golden.shape)
print("silver без golden leakage:", silver.shape)
print("golden clusters:", golden["cluster_id"].nunique())
print("silver clusters:", silver["cluster_id"].nunique())

clean_news: (99555, 14)
candidate_pool: (3176, 16)
golden: (121, 10)
silver без golden leakage: (1915, 10)
golden clusters: 34
silver clusters: 1736


## 3. Embeddings и сохранённая baseline-модель

Baseline использует `BAAI/bge-m3` и сохранённую CatBoost/fallback модель из предыдущего этапа.

In [49]:
baseline_config = SemanticNewsBaselineConfig()

encoder = SentenceTransformerEncoder(
    model_name=baseline_config.embedding_model_name,
    batch_size=16,
    normalize_embeddings=True,
    show_progress_bar=True,
)

current_model = CatBoostSignificanceModel.load(
    model_path=paths.current_catboost_model_path,
    config_path=paths.current_catboost_config_path,
)

# Приводим wrapper к feature set, на котором считались предыдущие метрики.
current_model.config = SignificanceModelConfig(
    threshold=0.42,
    duplicate_threshold=0.90,
    review_margin=0.10,
    feature_columns=LEGACY_SIGNIFICANCE_FEATURE_COLUMNS,
)
current_model.feature_columns = list(LEGACY_SIGNIFICANCE_FEATURE_COLUMNS)

tracker = ExperimentTracker(predictions_dir=paths.predictions_dir)
print(type(current_model.model))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Ignored unknown config keys: ['batch_size', 'depth', 'embeddings_cache_path', 'fallback_max_previous_candidates', 'fallback_similarity_threshold', 'fallback_window_days', 'first_item_fallback_enabled', 'iterations', 'l2_leaf_reg', 'learning_rate', 'local_model_dir', 'model_name', 'model_path', 'validation_size']
<class 'catboost.core.CatBoostClassifier'>


## 4. Baseline `exp_00b`

Воспроизводим reference pipeline: подготовка candidate pool → BGE-M3 embeddings → строгая threshold graph clustering → текущая novelty model.

In [50]:
legacy_pipeline = LegacyNewsNoveltyPipeline(
    encoder=encoder,
    novelty_model=current_model,
    config=LegacyNewsPipelineConfig(
        embeddings_cache_path=(
            paths.artifacts_dir / "embeddings" / "bge_m3_candidate_pool_id_aligned.npz"
        ),
        story_threshold=0.82,
        story_window_days=14,
        use_topic_blocking=True,
        normalize_embeddings_for_clustering=True,
    ),
)

legacy_result = legacy_pipeline.predict(raw_candidate_pool)

pred_00b = legacy_result.predictions
old_pipeline_clustered_news = legacy_result.clustered_news.reset_index(drop=True)
old_pipeline_embeddings = legacy_result.embeddings
baseline_cluster_ids = old_pipeline_clustered_news["cluster_id"].astype(str).reset_index(drop=True)

metrics_00b = tracker.register(
    experiment="exp_00b_reproduced_old_pipeline",
    golden=golden,
    prediction=pred_00b,
    embedding_variant="BAAI/bge-m3 id-aligned",
    clustering="baseline threshold graph: threshold=0.82, window=14",
    novelty_variant="current CatBoost/fallback model",
    comment="Воспроизводимый baseline из предыдущего этапа",
)

tracker.compact()

Рёбер в графе похожести: 914
Количество кластеров: 2607


,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
0,exp_00b_reproduced_old_pipeline,0.769737,0.0,0.724138,0.835616,0.835616,0.835616,Воспроизводимый baseline из предыдущего этапа


## 5. Выбор кластеризации без final-step классификации

Сначала сравниваем только качество кластеризации. Это быстрее и логичнее: final-step модель не должна маскировать ошибки clustering context.

In [51]:
candidate_pairs = build_candidate_pairs(
    old_pipeline_clustered_news,
    old_pipeline_embeddings,
    min_similarity=0.75,
    max_days=14,
)

print("candidate_pairs:", candidate_pairs.shape)

candidate_pairs: (2648, 10)


### 5.1. Diagnostic `exp_09b`

`09b` оставляем как промежуточный diagnostic experiment: он показывает, что conservative attach улучшает baseline. Но финально параметры выбираем не по golden, а через `exp_10`.

In [52]:
config_09b = AttachClusteringConfig(
    min_similarity=0.78,
    max_days=14,
    min_margin=0.03,
    source_max_cluster_size=2,
    require_evidence=True,
    title_jaccard_threshold=0.10,
    min_shared_numbers=1,
    cluster_prefix="exp09b",
)

cluster_ids_09b, diag_09b, attachments_09b = build_best_candidate_attach_clusters(
    old_pipeline_clustered_news,
    candidate_pairs,
    baseline_cluster_ids,
    config_09b,
)

metrics_cluster_00b = evaluate_cluster_ids_on_reference(
    golden,
    old_pipeline_clustered_news,
    baseline_cluster_ids,
)
metrics_cluster_09b = evaluate_cluster_ids_on_reference(
    golden,
    old_pipeline_clustered_news,
    cluster_ids_09b,
)

pd.DataFrame([
    {"experiment": "exp_00b", **metrics_cluster_00b},
    {"experiment": "exp_09b_diagnostic", **metrics_cluster_09b},
])[[
    "experiment", "tp_same_pairs", "fp_false_merge_pairs", "fn_missed_same_pairs",
    "pairwise_precision", "pairwise_recall", "pairwise_f1", "false_merge_rate",
]]

,experiment,tp_same_pairs,fp_false_merge_pairs,fn_missed_same_pairs,pairwise_precision,pairwise_recall,pairwise_f1,false_merge_rate
0,exp_00b,117,0,70,1.0,0.625668,0.769737,0.0
1,exp_09b_diagnostic,154,0,33,1.0,0.823529,0.903226,0.0


### 5.2. `exp_10`: выбор attach-параметров по silver-positive сигналу

Silver different-cluster пары не считаем надёжными negative. Используем только пары внутри одного silver-кластера как weak-positive сигнал и ограничиваем рост predicted pairs.

In [53]:
silver_for_clustering = prepare_silver_positive_reference(
    silver,
    golden,
    old_pipeline_clustered_news,
)

silver_cluster_sizes = silver_for_clustering["cluster_id"].value_counts()
print("Silver rows in candidate pool:", len(silver_for_clustering))
print("Silver clusters:", silver_for_clustering["cluster_id"].nunique())
print("Silver positive pairs:", int(silver_cluster_sizes.map(lambda x: x * (x - 1) // 2).sum()))
print("Silver singleton share:", float((silver_cluster_sizes == 1).mean()))

selection_config = SilverPositiveSelectionConfig(
    max_pred_pair_growth_over_baseline=1.55,
    max_all_data_cluster_size=80,
)

exp10_sweep, cluster_ids_by_variant_exp10, attachments_by_variant_exp10 = run_silver_positive_attach_sweep(
    news_df=old_pipeline_clustered_news,
    embeddings=old_pipeline_embeddings,
    silver_reference=silver_for_clustering,
    base_cluster_ids=baseline_cluster_ids,
    selection_config=selection_config,
    candidate_pairs=candidate_pairs,
)

best_exp10_row = select_silver_positive_variant(exp10_sweep, selection_config)
best_exp10_variant = str(best_exp10_row["variant"])
best_exp10_cluster_ids = cluster_ids_by_variant_exp10[best_exp10_variant]
best_exp10_config = get_attach_config_from_sweep_row(best_exp10_row)
attachments_exp10 = attachments_by_variant_exp10.get(best_exp10_variant, pd.DataFrame())

print("Выбранный exp_10 вариант:", best_exp10_variant)
print("Silver-positive recall:", best_exp10_row["silver_positive_recall"])
print("Silver predicted-pair growth:", best_exp10_row["silver_pred_pair_growth"])
print("Selected attachments:", len(attachments_exp10))

exp10_sweep.sort_values(
    ["silver_positive_recall", "silver_pred_pair_growth"],
    ascending=[False, True],
).head(20)

Silver rows in candidate pool: 1915
Silver clusters: 1736
Silver positive pairs: 242
Silver singleton share: 0.9141705069124424
Выбранный exp_10 вариант: exp10_src2_sim0.75_days7_m0.03_tj0.15_num1
Silver-positive recall: 0.7975206611570248
Silver predicted-pair growth: 1.5255198487712665
Selected attachments: 150


,variant,min_similarity,max_days,min_margin,source_max_cluster_size,title_jaccard_threshold,min_shared_numbers,require_evidence,candidate_attach_edges,attached_source_clusters,attached_rows,all_data_max_cluster_size,silver_positive_pairs,silver_recovered_positive_pairs,silver_missed_positive_pairs,silver_positive_recall,silver_total_pred_pairs,silver_pred_pair_growth,cluster_prefix,candidate_attach_targets
7,exp10_src2_sim0.75_days7_m0.03_tj0.05_num1,0.75,7.0,0.03,2.0,0.05,1.0,True,443,179,194,35,242,194,48,0.801653,1793,1.694707,exp10_src2_sim0.75_days7_m0.03_tj0.05_num1,229.0
11,exp10_src2_sim0.75_days7_m0.03_tj0.15_num1,0.75,7.0,0.03,2.0,0.15,1.0,True,323,150,165,33,242,193,49,0.797521,1614,1.525520,exp10_src2_sim0.75_days7_m0.03_tj0.15_num1,179.0
9,exp10_src2_sim0.75_days7_m0.03_tj0.10_num1,0.75,7.0,0.03,2.0,0.10,1.0,True,353,163,178,35,242,193,49,0.797521,1655,1.564272,exp10_src2_sim0.75_days7_m0.03_tj0.10_num1,198.0
56,exp10_src2_sim0.75_days10_m0.03_tj0.05_num2,0.75,10.0,0.03,2.0,0.05,2.0,True,439,183,200,35,242,193,49,0.797521,1798,1.699433,exp10_src2_sim0.75_days10_m0.03_tj0.05_num2,242.0
55,exp10_src2_sim0.75_days10_m0.03_tj0.05_num1,0.75,10.0,0.03,2.0,0.05,1.0,True,497,193,210,35,242,193,49,0.797521,1832,1.731569,exp10_src2_sim0.75_days10_m0.03_tj0.05_num1,255.0
104,exp10_src2_sim0.75_days14_m0.03_tj0.05_num2,0.75,14.0,0.03,2.0,0.05,2.0,True,500,200,217,36,242,193,49,0.797521,1897,1.793006,exp10_src2_sim0.75_days14_m0.03_tj0.05_num2,267.0
59,exp10_src2_sim0.75_days10_m0.03_tj0.15_num1,0.75,10.0,0.03,2.0,0.15,1.0,True,362,163,180,34,242,192,50,0.793388,1668,1.576560,exp10_src2_sim0.75_days10_m0.03_tj0.15_num1,200.0
57,exp10_src2_sim0.75_days10_m0.03_tj0.10_num1,0.75,10.0,0.03,2.0,0.10,1.0,True,394,176,193,35,242,192,50,0.793388,1690,1.597353,exp10_src2_sim0.75_days10_m0.03_tj0.10_num1,220.0
200,exp10_src2_sim0.76_days10_m0.03_tj0.05_num2,0.76,10.0,0.03,2.0,0.05,2.0,True,336,154,167,35,242,192,50,0.793388,1710,1.616257,exp10_src2_sim0.76_days10_m0.03_tj0.05_num2,191.0
199,exp10_src2_sim0.76_days10_m0.03_tj0.05_num1,0.76,10.0,0.03,2.0,0.05,1.0,True,381,162,176,35,242,192,50,0.793388,1727,1.632325,exp10_src2_sim0.76_days10_m0.03_tj0.05_num1,201.0


In [54]:
exp10_clustered_news = make_clustered_news(old_pipeline_clustered_news, best_exp10_cluster_ids)

metrics_cluster_10 = evaluate_cluster_ids_on_reference(
    golden,
    old_pipeline_clustered_news,
    best_exp10_cluster_ids,
)

cluster_comparison = pd.DataFrame([
    {"experiment": "exp_00b_baseline", **metrics_cluster_00b},
    {"experiment": "exp_09b_diagnostic", **metrics_cluster_09b},
    {"experiment": "exp_10_silver_positive", **metrics_cluster_10},
])

cluster_comparison[[
    "experiment", "tp_same_pairs", "fp_false_merge_pairs", "fn_missed_same_pairs",
    "pairwise_precision", "pairwise_recall", "pairwise_f1", "false_merge_rate",
]]

,experiment,tp_same_pairs,fp_false_merge_pairs,fn_missed_same_pairs,pairwise_precision,pairwise_recall,pairwise_f1,false_merge_rate
0,exp_00b_baseline,117,0,70,1.000000,0.625668,0.769737,0.000000
1,exp_09b_diagnostic,154,0,33,1.000000,0.823529,0.903226,0.000000
2,exp_10_silver_positive,156,3,31,0.981132,0.834225,0.901734,0.018868


## 6. Сравнение novelty classifiers на выбранной кластеризации

Кластеризация теперь фиксирована: `exp_10`. Дальше меняем только final-step модель.

In [55]:
exp10_features = build_legacy_significance_features(
    news_df=exp10_clustered_news,
    embeddings=old_pipeline_embeddings,
)
exp10_features["news_id"] = normalize_news_id(exp10_features["news_id"])

train_frame = make_significance_training_frame(
    features_df=exp10_features,
    labels_df=silver,
    eval_df=golden,
    feature_columns=LEGACY_SIGNIFICANCE_FEATURE_COLUMNS,
)

train_part, val_part = train_validation_split(
    train_frame,
    validation_size=0.25,
    random_state=42,
    group_column="cluster_id",
)

print("train_frame:", train_frame.shape)
print("train:", train_part.shape, "validation:", val_part.shape)
print("target distribution:")
print(train_frame["is_significant"].value_counts())

train_frame: (179, 24)
train: (140, 24) validation: (39, 24)
target distribution:
is_significant
1    133
0     46
Name: count, dtype: int64


In [56]:
def evaluate_exp10_model(experiment: str, model_wrapper, novelty_variant: str, comment: str):
    pred = model_wrapper.predict_clustered_with_fallback(
        news_df=exp10_clustered_news,
        embeddings=old_pipeline_embeddings,
    )
    metrics = tracker.register(
        experiment=experiment,
        golden=golden,
        prediction=pred,
        embedding_variant="BAAI/bge-m3 id-aligned",
        clustering=f"exp_10 silver-positive attach ({best_exp10_variant})",
        novelty_variant=novelty_variant,
        comment=comment,
    )
    return pred, metrics

# Контроль: текущая сохранённая модель на exp_10 кластеризации.
pred_10a, metrics_10a = evaluate_exp10_model(
    "exp_10a_current_model_on_exp10_clustering",
    current_model,
    "current CatBoost/fallback model",
    "Текущая сохранённая модель применена к выбранной exp_10 кластеризации",
)

tracker.compact().tail(3)

,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
0,exp_00b_reproduced_old_pipeline,0.769737,0.000000,0.724138,0.835616,0.835616,0.835616,Воспроизводимый baseline из предыдущего этапа
1,exp_10a_current_model_on_exp10_clustering,0.901734,0.018868,0.758621,0.842105,0.876712,0.859060,Текущая сохранённая модель применена к выбранн...


### 6.1. CatBoost / MLP / LogisticRegression

Все модели обучаются на silver features, построенных поверх уже выбранной `exp_10`-кластеризации. Golden исключён из train и используется только для оценки.

In [57]:
catboost_exp10_model, threshold_catboost = fit_catboost_binary(
    train_part,
    val_part,
    feature_columns=LEGACY_SIGNIFICANCE_FEATURE_COLUMNS,
    random_state=42,
)
pred_10b, metrics_10b = evaluate_exp10_model(
    "exp_10b_catboost_trained_on_exp10_clustering",
    catboost_exp10_model,
    "CatBoost trained on exp_10 features + fallback",
    "CatBoost обучен на silver features, построенных поверх выбранной exp_10 кластеризации",
)

threshold_catboost

0:	learn: 0.9116279	test: 0.9275362	best: 0.9275362 (0)	total: 1.48ms	remaining: 740ms
100:	learn: 0.9615385	test: 0.9253731	best: 0.9565217 (3)	total: 115ms	remaining: 455ms
200:	learn: 0.9900990	test: 0.9393939	best: 0.9565217 (3)	total: 229ms	remaining: 341ms
300:	learn: 1.0000000	test: 0.9393939	best: 0.9565217 (3)	total: 347ms	remaining: 229ms
400:	learn: 1.0000000	test: 0.9538462	best: 0.9565217 (3)	total: 460ms	remaining: 114ms
499:	learn: 1.0000000	test: 0.9538462	best: 0.9565217 (3)	total: 579ms	remaining: 0us

bestTest = 0.9565217391
bestIteration = 3

Shrink model to first 4 iterations.
0:	learn: 0.9175627	total: 1.42ms	remaining: 711ms
100:	learn: 0.9637681	total: 112ms	remaining: 444ms
200:	learn: 0.9815498	total: 229ms	remaining: 340ms
300:	learn: 0.9962547	total: 346ms	remaining: 229ms
400:	learn: 1.0000000	total: 466ms	remaining: 115ms
499:	learn: 1.0000000	total: 582ms	remaining: 0us


ThresholdSearchResult(threshold=0.49499999999999994, f1=0.9565217391304348, precision=0.9166666666666666, recall=1.0)

In [58]:
mlp_exp10_model, threshold_mlp = fit_mlp_binary(
    train_part,
    val_part,
    feature_columns=LEGACY_SIGNIFICANCE_FEATURE_COLUMNS,
    random_state=42,
)
pred_10c, metrics_10c = evaluate_exp10_model(
    "exp_10c_mlp_trained_on_exp10_clustering",
    mlp_exp10_model,
    "MLP trained on exp_10 features + fallback",
    "MLP обучена на silver features, построенных поверх выбранной exp_10 кластеризации",
)

threshold_mlp

ThresholdSearchResult(threshold=0.4499999999999999, f1=0.9428571428571428, precision=0.8918918918918919, recall=1.0)

In [59]:
logreg_exp10_model, threshold_logreg = fit_logreg_binary(
    train_part,
    val_part,
    feature_columns=LEGACY_SIGNIFICANCE_FEATURE_COLUMNS,
    random_state=42,
)
pred_10d, metrics_10d = evaluate_exp10_model(
    "exp_10d_logreg_trained_on_exp10_clustering",
    logreg_exp10_model,
    "LogisticRegression trained on exp_10 features + fallback",
    "LogisticRegression обучена на silver features, построенных поверх выбранной exp_10 кластеризации",
)

threshold_logreg

ThresholdSearchResult(threshold=0.195, f1=0.9565217391304348, precision=0.9166666666666666, recall=1.0)

### 6.2. Опциональная проверка transfer-модели из старых экспериментов

Если в памяти есть `mlp_04b_model`, можно сравнить её как transfer-вариант. Эта ячейка не обязательна для воспроизводимости v3.

In [60]:
if "mlp_04b_model" in globals():
    pred_10_transfer_mlp, metrics_10_transfer_mlp = evaluate_exp10_model(
        "exp_10e_legacy_mlp_transfer_on_exp10_clustering",
        mlp_04b_model,
        "legacy MLP from exp_04b + fallback",
        "MLP из exp_04b применена к exp_10 кластеризации без переобучения",
    )
else:
    print("mlp_04b_model не найден в памяти; transfer-check пропущен.")

tracker.compact().sort_values("significant_f1", ascending=False)

mlp_04b_model не найден в памяти; transfer-check пропущен.


,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
1,exp_10a_current_model_on_exp10_clustering,0.901734,0.018868,0.758621,0.842105,0.876712,0.859060,Текущая сохранённая модель применена к выбранн...
3,exp_10c_mlp_trained_on_exp10_clustering,0.901734,0.018868,0.747126,0.831169,0.876712,0.853333,"MLP обучена на silver features, построенных по..."
0,exp_00b_reproduced_old_pipeline,0.769737,0.000000,0.724138,0.835616,0.835616,0.835616,Воспроизводимый baseline из предыдущего этапа
4,exp_10d_logreg_trained_on_exp10_clustering,0.901734,0.018868,0.712644,0.833333,0.821918,0.827586,"LogisticRegression обучена на silver features,..."
2,exp_10b_catboost_trained_on_exp10_clustering,0.901734,0.018868,0.689655,0.848485,0.767123,0.805755,"CatBoost обучен на silver features, построенны..."


## 7. Выбор и экспорт финальной модели

Выбираем лучшую модель среди `exp_10*` по `significant_f1`, при равенстве — по `significant_recall`. При желании можно вручную переопределить `SELECTED_MODEL_EXPERIMENT`.

In [61]:
model_selection_table = tracker.compact().copy()
model_selection_table = model_selection_table[
    model_selection_table["experiment"].str.startswith("exp_10", na=False)
].sort_values(
    ["significant_f1", "significant_recall", "significant_precision"],
    ascending=[False, False, False],
).reset_index(drop=True)

model_selection_table

,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
0,exp_10a_current_model_on_exp10_clustering,0.901734,0.018868,0.758621,0.842105,0.876712,0.859060,Текущая сохранённая модель применена к выбранн...
1,exp_10c_mlp_trained_on_exp10_clustering,0.901734,0.018868,0.747126,0.831169,0.876712,0.853333,"MLP обучена на silver features, построенных по..."
2,exp_10d_logreg_trained_on_exp10_clustering,0.901734,0.018868,0.712644,0.833333,0.821918,0.827586,"LogisticRegression обучена на silver features,..."
3,exp_10b_catboost_trained_on_exp10_clustering,0.901734,0.018868,0.689655,0.848485,0.767123,0.805755,"CatBoost обучен на silver features, построенны..."


In [62]:
SELECTED_MODEL_EXPERIMENT = model_selection_table.iloc[0]["experiment"]
print("Selected final model experiment:", SELECTED_MODEL_EXPERIMENT)

selected_model_objects = {
    "exp_10a_current_model_on_exp10_clustering": current_model,
    "exp_10b_catboost_trained_on_exp10_clustering": catboost_exp10_model,
    "exp_10c_mlp_trained_on_exp10_clustering": mlp_exp10_model,
    "exp_10d_logreg_trained_on_exp10_clustering": logreg_exp10_model,
}
if "mlp_04b_model" in globals():
    selected_model_objects["exp_10e_legacy_mlp_transfer_on_exp10_clustering"] = mlp_04b_model

selected_model = selected_model_objects[SELECTED_MODEL_EXPERIMENT]

final_model_dir = paths.models_dir / "final_exp10"
final_model_dir.mkdir(parents=True, exist_ok=True)
final_model_path = final_model_dir / f"{SELECTED_MODEL_EXPERIMENT}.joblib"
joblib.dump(selected_model, final_model_path)

# Сохраняем config финального pipeline с выбранными exp_10 параметрами.
from final_pipeline import FinalPipelineConfig

final_config = FinalPipelineConfig(attach_clustering=best_exp10_config)
final_config_path = final_model_dir / "final_pipeline_config.json"
final_config.to_json(final_config_path)

print("Saved model:", final_model_path)
print("Saved pipeline config:", final_config_path)

Selected final model experiment: exp_10a_current_model_on_exp10_clustering
Saved model: E:\ML\Projects\Git\news-flow-analysis\data\artifacts\models\final_exp10\exp_10a_current_model_on_exp10_clustering.joblib
Saved pipeline config: E:\ML\Projects\Git\news-flow-analysis\data\artifacts\models\final_exp10\final_pipeline_config.json


## 8. Опциональная проверка дообученного BGE-M3

Эта проверка не участвует в выборе параметров. Она нужна как ablation: меняется только embedding model/cache, кластеризация остаётся `exp_10`, novelty model — выбранная выше.

In [63]:
FINETUNED_BGE_MODEL_CANDIDATES = [
    paths.models_dir / "news-flow-bge-m3-ru-paraphrase-mnrl" / "final",
    paths.models_dir / "news-flow-bge-m3-ru-paraphrase-mnrl" / "checkpoint-95",
    paths.models_dir / "bge_m3_finetuned",
]
FINETUNED_BGE_MODEL_PATH = next(
    (path for path in FINETUNED_BGE_MODEL_CANDIDATES if path.exists()),
    FINETUNED_BGE_MODEL_CANDIDATES[0],
)

FINETUNED_EMBEDDINGS_CACHE_CANDIDATES = [
    paths.artifacts_dir / "embeddings" / "exp_08_finetuned_bge_m3_candidate_pool_embeddings.npz"
]
FINETUNED_EMBEDDINGS_CACHE = next(
    (path for path in FINETUNED_EMBEDDINGS_CACHE_CANDIDATES if path.exists()),
    FINETUNED_EMBEDDINGS_CACHE_CANDIDATES[-1],
)

print("FINETUNED_BGE_MODEL_PATH:", FINETUNED_BGE_MODEL_PATH)
print("FINETUNED_EMBEDDINGS_CACHE:", FINETUNED_EMBEDDINGS_CACHE)

if FINETUNED_BGE_MODEL_PATH.exists():
    import gc

    # Keep only one SentenceTransformer model resident while running the optional ablation.
    if "encoder" in globals():
        del encoder
        gc.collect()

    finetuned_encoder = SentenceTransformerEncoder(
        model_name=str(FINETUNED_BGE_MODEL_PATH),
        batch_size=16,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    from model.embeddings import get_or_create_id_aligned_embeddings
    from model.attach_clustering import build_baseline_cluster_ids

    finetuned_embeddings = get_or_create_id_aligned_embeddings(
        encoder=finetuned_encoder,
        df=old_pipeline_clustered_news,
        cache_path=FINETUNED_EMBEDDINGS_CACHE,
        id_column="news_id",
        text_column="model_text",
    )
    finetuned_base_ids, _ = build_baseline_cluster_ids(
        old_pipeline_clustered_news,
        finetuned_embeddings,
        BaselineClusteringConfig(),
    )
    finetuned_pairs = build_candidate_pairs(
        old_pipeline_clustered_news,
        finetuned_embeddings,
        min_similarity=best_exp10_config.min_similarity,
        max_days=max(best_exp10_config.max_days, 14),
    )
    finetuned_ids, _, _ = build_best_candidate_attach_clusters(
        old_pipeline_clustered_news,
        finetuned_pairs,
        finetuned_base_ids,
        best_exp10_config,
    )
    finetuned_clustered_news = make_clustered_news(old_pipeline_clustered_news, finetuned_ids)
    pred_finetuned = selected_model.predict_clustered_with_fallback(
        news_df=finetuned_clustered_news,
        embeddings=finetuned_embeddings,
    )
    tracker.register(
        experiment="exp_11_finetuned_bge_with_selected_exp10_model",
        golden=golden,
        prediction=pred_finetuned,
        embedding_variant="fine-tuned BGE-M3",
        clustering="same exp_10 attach config",
        novelty_variant=SELECTED_MODEL_EXPERIMENT,
        comment="Ablation: проверка дообученного BGE-M3 без повторного подбора параметров",
    )
else:
    print(f"Дообученный BGE-M3 не найден: {FINETUNED_BGE_MODEL_PATH}")
    print("Ячейка пропущена. Укажите актуальный путь в FINETUNED_BGE_MODEL_PATH, если модель есть.")

FINETUNED_BGE_MODEL_PATH: E:\ML\Projects\Git\news-flow-analysis\data\artifacts\models\news-flow-bge-m3-ru-paraphrase-mnrl\checkpoint-95
FINETUNED_EMBEDDINGS_CACHE: E:\ML\Projects\Git\news-flow-analysis\data\artifacts\embeddings\exp_08_finetuned_bge_m3_candidate_pool_embeddings.npz


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Рёбер в графе похожести: 828
Количество кластеров: 2639


## 9. Финальное сравнение с baseline

In [64]:
final_results = tracker.compact().copy()
final_results.sort_values(
    ["significant_f1", "significant_recall", "pairwise_f1"],
    ascending=[False, False, False],
)

,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
1,exp_10a_current_model_on_exp10_clustering,0.901734,0.018868,0.758621,0.842105,0.876712,0.859060,Текущая сохранённая модель применена к выбранн...
3,exp_10c_mlp_trained_on_exp10_clustering,0.901734,0.018868,0.747126,0.831169,0.876712,0.853333,"MLP обучена на silver features, построенных по..."
0,exp_00b_reproduced_old_pipeline,0.769737,0.000000,0.724138,0.835616,0.835616,0.835616,Воспроизводимый baseline из предыдущего этапа
4,exp_10d_logreg_trained_on_exp10_clustering,0.901734,0.018868,0.712644,0.833333,0.821918,0.827586,"LogisticRegression обучена на silver features,..."
5,exp_11_finetuned_bge_with_selected_exp10_model,0.808777,0.022727,0.701149,0.821918,0.821918,0.821918,Ablation: проверка дообученного BGE-M3 без пов...
2,exp_10b_catboost_trained_on_exp10_clustering,0.901734,0.018868,0.689655,0.848485,0.767123,0.805755,"CatBoost обучен на silver features, построенны..."


In [65]:
baseline_row = final_results[final_results["experiment"].eq("exp_00b_reproduced_old_pipeline")].iloc[0]
selected_row = final_results[final_results["experiment"].eq(SELECTED_MODEL_EXPERIMENT)].iloc[0]

compare_columns = [
    "pairwise_f1",
    "false_merge_rate",
    "significance_accuracy",
    "significant_precision",
    "significant_recall",
    "significant_f1",
]

baseline_vs_selected = pd.DataFrame({
    "metric": compare_columns,
    "baseline_00b": [baseline_row[column] for column in compare_columns],
    "selected_exp10": [selected_row[column] for column in compare_columns],
})
baseline_vs_selected["delta"] = baseline_vs_selected["selected_exp10"] - baseline_vs_selected["baseline_00b"]
baseline_vs_selected

,metric,baseline_00b,selected_exp10,delta
0,pairwise_f1,0.769737,0.901734,0.131997
1,false_merge_rate,0.000000,0.018868,0.018868
2,significance_accuracy,0.724138,0.758621,0.034483
3,significant_precision,0.835616,0.842105,0.006489
4,significant_recall,0.835616,0.876712,0.041096
5,significant_f1,0.835616,0.859060,0.023444


## 10. Выводы

1. Baseline `exp_00b` был зафиксирован как воспроизводимая контрольная точка: BGE-M3 embeddings, строгая topic-aware temporal graph clustering и сохранённая CatBoost/fallback novelty model.
2. Основное ограничение baseline — недосклейка сюжетов: высокая precision кластеризации достигается ценой низкого recall same-story пар.
3. Silver-разметка оказалась пере-фрагментированной и не подходит как полноценный источник negative pairs для story clustering. Поэтому в `exp_10` она использована только как weak-positive сигнал.
4. `exp_10` выбирает параметры attach-кластеризации без human golden: максимизирует silver-positive recall при ограничении роста predicted pairs и размера кластеров.
5. После фиксации `exp_10`-кластеризации отдельно сравниваются final-step модели novelty detection. Такой порядок разделяет две задачи: сначала clustering context, потом classification.
6. Финальная модель сохраняется в `data/artifacts/models/final_exp10/` и используется тем же кодом, что и final pipeline.
7. Diagnostic upper-bound эксперименты, подобранные напрямую по golden, можно упоминать отдельно, но основной честный improvement — `exp_10` + выбранная novelty model.

In [66]:
# ---------------------------------------------------------------------
# Diagnostic: воспроизводятся ли старые параметры exp_10 в v3
# ---------------------------------------------------------------------

old_exp10_config = AttachClusteringConfig(
    min_similarity=0.75,
    max_days=7,
    min_margin=0.03,
    source_max_cluster_size=2,
    require_evidence=True,
    title_jaccard_threshold=0.15,
    min_shared_numbers=1,
    cluster_prefix="old_exp10_fixed_check",
)

old_exp10_cluster_ids, old_exp10_diagnostics, old_exp10_attachments = (
    build_best_candidate_attach_clusters(
        old_pipeline_clustered_news,
        candidate_pairs,
        baseline_cluster_ids,
        old_exp10_config,
    )
)

old_exp10_clustered_news = make_clustered_news(
    old_pipeline_clustered_news,
    old_exp10_cluster_ids,
)

old_exp10_metrics_clustering = evaluate_cluster_ids_on_reference(
    golden,
    old_pipeline_clustered_news,
    old_exp10_cluster_ids,
)

old_exp10_pred = current_model.predict_clustered_with_fallback(
    news_df=old_exp10_clustered_news,
    embeddings=old_pipeline_embeddings,
)

old_exp10_metrics = evaluate_predictions(
    golden,
    old_exp10_pred,
)

print("Old exp_10 config:")
print(old_exp10_config)

print("\nOld exp_10 diagnostics:")
print(old_exp10_diagnostics)

print("\nOld exp_10 clustering metrics:")
display(pd.DataFrame([old_exp10_metrics_clustering])[
    [
        "tp_same_pairs",
        "fp_false_merge_pairs",
        "fn_missed_same_pairs",
        "pairwise_precision",
        "pairwise_recall",
        "pairwise_f1",
        "false_merge_rate",
    ]
])

print("\nOld exp_10 end-to-end metrics:")
display(pd.DataFrame([old_exp10_metrics])[
    [
        "pairwise_f1",
        "false_merge_rate",
        "significance_accuracy",
        "significant_precision",
        "significant_recall",
        "significant_f1",
    ]
])

Old exp_10 config:
AttachClusteringConfig(min_similarity=0.75, max_days=7, min_margin=0.03, source_max_cluster_size=2, require_evidence=True, title_jaccard_threshold=0.15, min_shared_numbers=1, cluster_prefix='old_exp10_fixed_check')

Old exp_10 diagnostics:
{'candidate_attach_edges': 323, 'candidate_attach_targets': 179, 'attached_source_clusters': 150, 'attached_rows': 165}

Old exp_10 clustering metrics:


,tp_same_pairs,fp_false_merge_pairs,fn_missed_same_pairs,pairwise_precision,pairwise_recall,pairwise_f1,false_merge_rate
0,156,3,31,0.981132,0.834225,0.901734,0.018868



Old exp_10 end-to-end metrics:


,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1
0,0.901734,0.018868,0.758621,0.842105,0.876712,0.85906


In [67]:
# ---------------------------------------------------------------------
# Что реально выбрал exp_10 в v3
# ---------------------------------------------------------------------

print("best_exp10_variant:", best_exp10_variant)
print("\nbest_exp10_config:")
print(best_exp10_config)

display(pd.DataFrame([best_exp10_row]))

best_exp10_variant: exp10_src2_sim0.75_days7_m0.03_tj0.15_num1

best_exp10_config:
AttachClusteringConfig(min_similarity=0.75, max_days=7, min_margin=0.03, source_max_cluster_size=2, require_evidence=True, title_jaccard_threshold=0.15, min_shared_numbers=1, cluster_prefix='exp10_src2_sim0.75_days7_m0.03_tj0.15_num1')


,variant,min_similarity,max_days,min_margin,source_max_cluster_size,title_jaccard_threshold,min_shared_numbers,require_evidence,candidate_attach_edges,attached_source_clusters,attached_rows,all_data_max_cluster_size,silver_positive_pairs,silver_recovered_positive_pairs,silver_missed_positive_pairs,silver_positive_recall,silver_total_pred_pairs,silver_pred_pair_growth,cluster_prefix,candidate_attach_targets
0,exp10_src2_sim0.75_days7_m0.03_tj0.15_num1,0.75,7.0,0.03,2.0,0.15,1.0,True,323,150,165,33,242,193,49,0.797521,1614,1.52552,exp10_src2_sim0.75_days7_m0.03_tj0.15_num1,179.0
